# A2 — Features: ESM-2, chemistry, structure and the family frame (Stage 2)

**Question.** How is each variant turned into numbers a model can learn from, and where does it sit
in 3D?

**What it does.**
- One table of every variant (ddPCA, MegaScale, atlas) with the **wild-type sequence** of its
  domain; every row passes a residue check.
- **ESM-2 650M, one pass per wild-type domain** (725 domains): the 1,280-number representation of
  each residue plus masked-marginal log-odds, so every variant's features are a lookup
  (5 min instead of hours; 97 MB instead of 2 GB).
- Seven amino-acid chemistry scalars. **1,292 numbers per variant in total.**
- Structure: the three solved ddPCA complexes, and AlphaFold models for the atlas proteins
  (downloaded once, then **cached on Drive**). Structure is annotation for validation only,
  never a model input.
- Every atlas residue placed on the GRB2-SH3 reference frame (56 positions).

**Inputs:** A0 atlas, A1 ground truth, `input/datasets/Data.zip` (PDB files), `input/uniprot/sh3_domains_human.tsv`.
**Outputs:** `features_{megascale,ddpca,atlas}.parquet`, `esm_domain_embeddings.npz`,
`structure_features.csv`, `alphafold_log.csv`, `RESULTS_A2.md`.
**Runtime:** about 25 min on a **T4 GPU** (15 of them AlphaFold downloads on the first run only).
Explanation: `docs/A2_features.md`.

In [ ]:
# ── setup (identical in every notebook) ─────────────────────────────────────────
ROOT   = '/content/drive/MyDrive/sh3_repro'   # the Drive folder from UPLOAD_GUIDE.md
RESUME = True                                  # reuse finished steps after a disconnect

import os, sys, json, time, importlib
from google.colab import drive; drive.mount('/content/drive')
os.system('pip -q install biopython')
sys.path.insert(0, f'{ROOT}/code')
import sh3_common as C; importlib.reload(C)
import numpy as np, pandas as pd
R = C.Run(ROOT, 'A2', 'Features', resume=RESUME)
if C.DEV == 'cpu':
    print('WARNING: no GPU. Runtime -> Change runtime type -> T4 GPU.')
print(C.env_info())

In [ ]:
import collections, urllib.request, shutil
import torch
from Bio import Align
from Bio.Align import substitution_matrices
from Bio.PDB import MMCIFParser
from Bio.PDB.SASA import ShrakeRupley
R.verify_inputs(['input/datasets/Data.zip', 'input/uniprot/sh3_domains_human.tsv'])
R.extract('input/datasets/Data.zip', lambda s: s.startswith('Data/pdb/'))
W = R.WORK; PAT = C.MUT_PATTERN
FEAT = ['features_megascale.parquet', 'features_ddpca.parquet', 'features_atlas.parquet', 'structure_features.csv']
dd = pd.read_parquet(R.res('A1', 'ddpca_ground_truth.parquet'))
mega = pd.read_parquet(R.res('A1', 'megascale_singles.parquet'))
MASTER = pd.read_csv(R.res('A0', 'SH3_cancer_master.tsv'), sep='\t')
DOMTBL = pd.read_csv(f'{R.IN}/uniprot/sh3_domains_human.tsv', sep='\t')
print(f'ddpca {len(dd):,} | megascale {len(mega):,} | atlas {len(MASTER):,}')

## 1. One variant table, with the wild-type sequence of every domain
MegaScale's `aa_seq` is the **mutant** sequence (a first-run bug in the original project); the wild type is reconstructed.

In [ ]:
frames = []
for dom, g in dd.groupby('domain'):     # ddPCA: consensus wild-type residue at each position
    seq = ''.join(g.groupby('pos').wt_aa.agg(lambda s: s.mode().iat[0]).sort_index().values)
    sub = g[['pos', 'wt_aa', 'mut_aa', 'folding_coefficient', 'binding_coefficient', 'binding_reliable']].copy()
    frames.append(sub.rename(columns={'folding_coefficient': 'ddG_fold', 'binding_coefficient': 'ddG_bind'})
                     .assign(set='ddpca', domain_key=dom, wt_seq=seq))
x = mega.mut_type.str.extract(PAT)
ms = mega.assign(wt_aa=x[0], pos=pd.to_numeric(x[1]), mut_aa=x[2]).dropna(subset=['pos', 'wt_aa', 'mut_aa', 'aa_seq'])
ms['pos'] = ms.pos.astype(int)
mutant_check = float(np.mean([s[p-1] == m for s, p, m in zip(ms.aa_seq, ms.pos, ms.mut_aa)]))
ms['wt_seq'] = [s[:p-1] + w + s[p:] for s, p, w in zip(ms.aa_seq, ms.pos, ms.wt_aa)]
frames.append(ms[['WT_name', 'pos', 'wt_aa', 'mut_aa', 'ddG_fold', 'is_sh3', 'gene', 'uniprot', 'domain_idx', 'wt_seq']]
              .rename(columns={'WT_name': 'domain_key'}).assign(set='megascale'))
at = MASTER[['variant_id', 'gene', 'uniprot', 'domain_idx', 'wt_domain_seq', 'pos_in_domain', 'wt_aa', 'mut_aa']]
at = at.rename(columns={'wt_domain_seq': 'wt_seq', 'pos_in_domain': 'pos'})
frames.append(at.assign(domain_key=at.uniprot + '_d' + at.domain_idx.astype(str), set='atlas'))
V = pd.concat(frames, ignore_index=True); V['pos'] = V.pos.astype('Int64')
n0 = len(V); V = V.dropna(subset=['wt_seq', 'pos', 'wt_aa', 'mut_aa'])
V = V[[len(s) >= p and s[p-1] == w for s, p, w in zip(V.wt_seq, V.pos, V.wt_aa)]].reset_index(drop=True)
DROPPED = n0 - len(V)
V['dom_len'] = V.wt_seq.str.len(); V['rel_pos'] = V.pos / V.dom_len
B62 = substitution_matrices.load('BLOSUM62')
V['d_hydropathy'] = V.mut_aa.map(C.KD) - V.wt_aa.map(C.KD)
V['d_volume'] = V.mut_aa.map(C.VOL) - V.wt_aa.map(C.VOL)
V['d_charge'] = V.mut_aa.map(C.CHG).fillna(0) - V.wt_aa.map(C.CHG).fillna(0)
V['blosum62'] = [B62[a][b] if (a in B62.alphabet and b in B62.alphabet) else np.nan for a, b in zip(V.wt_aa, V.mut_aa)]
V['to_proline'] = (V.mut_aa == 'P').astype(int); V['from_glycine'] = (V.wt_aa == 'G').astype(int)
V['to_glycine'] = (V.mut_aa == 'G').astype(int)
print(f'MegaScale aa_seq carries the mutant residue for {mutant_check:.1%} of rows (confirms it is the mutant sequence)')
print(f'variant table: {len(V):,} rows, {DROPPED} dropped by the residue check', V.groupby('set').size().to_dict())
assert DROPPED == 0, 'rows failed the residue check - investigate before continuing'

## 2. ESM-2: one forward pass per wild-type domain

In [ ]:
if R.done('esm_domain_embeddings.npz', 'esm_scores.parquet', 'esm_logodds.npz'):
    EMB = C.load_embeddings(R.out('esm_domain_embeddings.npz'))
    V = V.join(pd.read_parquet(R.out('esm_scores.parquet'))); print('ESM features loaded from this folder')
else:
    from transformers import AutoTokenizer, AutoModelForMaskedLM
    tok = AutoTokenizer.from_pretrained('facebook/esm2_t33_650M_UR50D')
    esm = AutoModelForMaskedLM.from_pretrained('facebook/esm2_t33_650M_UR50D').to(C.DEV).eval()
    if C.DEV == 'cuda': esm = esm.half()
    AAS = list('ACDEFGHIKLMNPQRSTVWY'); AAPOS = {a: i for i, a in enumerate(AAS)}
    AA_IDX = torch.tensor([tok.convert_tokens_to_ids(a) for a in AAS]).to(C.DEV)
    @torch.no_grad()
    def embed_and_score(seq, batch=32):
        L = len(seq); enc = tok(seq, return_tensors='pt').to(C.DEV)
        rep = esm(**enc, output_hidden_states=True).hidden_states[-1][0, 1:L+1].float().cpu().numpy()
        ids = enc['input_ids'][0]; rows = []
        for s in range(0, L, batch):
            idx = list(range(s, min(s + batch, L)))
            bb = ids.unsqueeze(0).repeat(len(idx), 1).clone()
            for k, i in enumerate(idx): bb[k, i + 1] = tok.mask_token_id        # residue i is token i+1
            lg = esm(input_ids=bb, attention_mask=torch.ones_like(bb)).logits
            for k, i in enumerate(idx): rows.append(torch.log_softmax(lg[k, i + 1].float(), -1)[AA_IDX].cpu().numpy())
        return rep, np.stack(rows)
    seqs = V[['domain_key', 'wt_seq']].drop_duplicates('domain_key')
    EMB, LO = {}, {}; t0 = time.time()
    for n, (k, s) in enumerate(zip(seqs.domain_key, seqs.wt_seq), 1):
        e, l = embed_and_score(s); EMB[k] = e.astype(np.float16); LO[k] = l
        if n % 100 == 0: print(f'  ESM {n}/{len(seqs)}  {(time.time() - t0) / 60:.1f} min')
    lo = np.stack([LO[k][p-1] for k, p in zip(V.domain_key, V.pos.astype(int))]); r = np.arange(len(V))
    w = V.wt_aa.map(AAPOS).values; mu = V.mut_aa.map(AAPOS).values
    V['esm_score'] = lo[r, mu] - lo[r, w]; V['esm_wt_logp'] = lo[r, w]; V['esm_entropy'] = -(np.exp(lo) * lo).sum(1)
    del lo, esm; torch.cuda.empty_cache()
    np.savez_compressed(R.out('esm_domain_embeddings.npz'), **EMB)
    np.savez_compressed(R.out('esm_logodds.npz'), **{k: v.astype(np.float32) for k, v in LO.items()})   # for B6
    V[['esm_score', 'esm_wt_logp', 'esm_entropy']].to_parquet(R.out('esm_scores.parquet'))
print(f'{len(EMB)} domains embedded')

## 3. Structure (annotation only)
Solved complexes for the ddPCA domains, with the PDB→construct offset derived by sliding the construct along the chain. AlphaFold models for the atlas, cached in `cache/alphafold/`. Relative solvent accessibility by Shrake–Rupley, normalised by a standard maximum-accessibility table (the table's primary source could not be confirmed; see the doc).

In [ ]:
T3 = {'ALA':'A','ARG':'R','ASN':'N','ASP':'D','CYS':'C','GLN':'Q','GLU':'E','GLY':'G','HIS':'H','ILE':'I',
      'LEU':'L','LYS':'K','MET':'M','PHE':'F','PRO':'P','SER':'S','THR':'T','TRP':'W','TYR':'Y','VAL':'V'}
def struct_from_pdb(pdb, rsasa_pdb, dom_chain='A', pep_chain='B'):
    rs = {int(l[22:26]): float(l[60:66]) for l in open(rsasa_pdb) if l.startswith('ATOM') and l[21] == dom_chain}
    dom, pep = collections.defaultdict(list), []
    for l in open(pdb):
        if not l.startswith('ATOM'): continue
        xyz = (float(l[30:38]), float(l[38:46]), float(l[46:54]))
        if l[21] == dom_chain: dom[int(l[22:26])].append(xyz)
        elif l[21] == pep_chain: pep.append(xyz)
    return pd.DataFrame([{'pos': r, 'rsasa': rs.get(r, np.nan),
                          'dist_peptide': min(((a[0]-b[0])**2 + (a[1]-b[1])**2 + (a[2]-b[2])**2) ** .5
                                              for a in dom[r] for b in pep) if pep else np.nan} for r in sorted(dom)])
def derive_offset(construct, pdb, ch='A'):
    seen = {}
    for l in open(pdb):
        if l.startswith('ATOM') and l[21] == ch: seen.setdefault(int(l[22:26]), T3.get(l[17:20].strip(), 'X'))
    ks = sorted(seen); seq = ''.join(seen[k] for k in ks); best = (0, None)
    for k in range(max(1, len(seq) - len(construct) + 1)):
        ident = sum(a == b for a, b in zip(construct, seq[k:]))
        if ident > best[0]: best = (ident, k)
    frac = best[0] / len(construct)
    return (ks[best[1]] - 1, frac) if (best[1] is not None and frac >= 0.8) else (None, frac)

struct, offsets = [], []
for dom, pid in {'GRB2-SH3': '2vwf', 'PSD95-PDZ3': '1be9', 'GB1': '2gb1'}.items():
    construct = ''.join(V[V.domain_key == dom].groupby('pos').wt_aa.first().sort_index().values)
    off, frac = derive_offset(construct, f'{W}/Data/pdb/{pid}.pdb')
    offsets.append(dict(domain=dom, pdb=pid, offset=off, identity=round(frac, 3)))
    if off is None: continue
    s = struct_from_pdb(f'{W}/Data/pdb/{pid}.pdb', f'{W}/Data/pdb/{pid}_RSASA.pdb')
    s['pos'] = s['pos'] - off
    struct.append(s[(s.pos >= 1) & (s.pos <= len(construct))].assign(domain_key=dom, plddt=np.nan))
print(pd.DataFrame(offsets).to_string(index=False))

MAXACC = {'A':106,'R':248,'N':157,'D':163,'C':135,'Q':198,'E':194,'G':84,'H':184,'I':169,
          'L':164,'K':205,'M':188,'F':197,'P':136,'S':130,'T':142,'W':227,'Y':222,'V':142}
AF = f'{R.CACHE}/alphafold'; os.makedirs(AF, exist_ok=True)
sr = ShrakeRupley(); aflog = []
def alphafold(acc, a, b):
    f = f'{AF}/AF-{acc}.cif'; ver = 'cache'
    if not os.path.exists(f):
        ver = None
        for v in ('v6', 'v5', 'v4'):
            try:
                urllib.request.urlretrieve(f'https://alphafold.ebi.ac.uk/files/AF-{acc}-F1-model_{v}.cif', f + '.part')
                shutil.move(f + '.part', f); ver = v; break
            except Exception:
                pass
        if ver is None: aflog.append(dict(uniprot=acc, status='not available')); return None
    try:
        ch = next(MMCIFParser(QUIET=True).get_structure(acc, f)[0].get_chains()); sr.compute(ch, level='R')
        aflog.append(dict(uniprot=acc, status=ver))
        return pd.DataFrame([{'pos': x.id[1] - a + 1, 'rsasa': x.sasa / MAXACC.get(T3.get(x.get_resname(), 'X'), 200),
                              'dist_peptide': np.nan, 'plddt': np.mean([t.get_bfactor() for t in x])}
                             for x in ch if a <= x.id[1] <= b])
    except Exception as e:
        aflog.append(dict(uniprot=acc, status=f'parse error: {type(e).__name__}')); return None
todo = DOMTBL.assign(domain_key=DOMTBL.uniprot + '_d' + DOMTBL.domain_idx.astype(str))
todo = todo[todo.domain_key.isin(set(V[V.set == 'atlas'].domain_key))]; t0 = time.time()
for n, (_, d) in enumerate(todo.iterrows(), 1):
    s = alphafold(d.uniprot, int(d.start), int(d.end))
    if s is not None and len(s): struct.append(s.assign(domain_key=d.domain_key))
    if n % 50 == 0: print(f'  AlphaFold {n}/{len(todo)}  {(time.time() - t0) / 60:.1f} min')
ST = pd.concat(struct, ignore_index=True)
AFL = pd.DataFrame(aflog).drop_duplicates(); AFL.to_csv(R.out('alphafold_log.csv'), index=False)
AF_FAILED = sorted(set(AFL[~AFL.status.isin(['v6', 'v5', 'v4', 'cache'])].uniprot))
print(f'structure: {len(ST):,} rows over {ST.domain_key.nunique()} domains | proteins without a model: {AF_FAILED}')

## 4. Every atlas residue on the GRB2-SH3 reference frame, then save

In [ ]:
al = Align.PairwiseAligner(mode='global', open_gap_score=-11, extend_gap_score=-1); al.match_score = 2; al.mismatch_score = -1
ref_seq = V[V.domain_key == 'GRB2-SH3'].wt_seq.iat[0]
def to_ref(seq):
    try: a = al.align(ref_seq, seq)[0]
    except Exception: return {}
    return {qs + k + 1: rs_ + k + 1 for (rs_, re_), (qs, qe) in zip(a.aligned[0], a.aligned[1]) for k in range(re_ - rs_)}
seqs = V[['domain_key', 'wt_seq']].drop_duplicates('domain_key')
atlas_keys = set(V[V.set == 'atlas'].domain_key)
REFMAP = {k: to_ref(s) for k, s in zip(seqs.domain_key, seqs.wt_seq) if k in atlas_keys}
V['ref_pos'] = [REFMAP[k].get(int(p), np.nan) if k in REFMAP else np.nan for k, p in zip(V.domain_key, V.pos)]
V = V.merge(ST[['domain_key', 'pos', 'rsasa', 'dist_peptide']], on=['domain_key', 'pos'], how='left')
V['has_structure'] = V.rsasa.notna()
V['buried'] = (V.rsasa < 0.25).where(V.rsasa.notna()); V['interface'] = (V.dist_peptide < 5.0).where(V.dist_peptide.notna())
MG, DD, AT = [V[V.set == s].drop(columns=['set']).reset_index(drop=True) for s in ('megascale', 'ddpca', 'atlas')]
for d, f in zip((MG, DD, AT), FEAT[:3]): d.to_parquet(R.out(f), index=False)
ST.to_csv(R.out('structure_features.csv'), index=False)
print(f'megascale {len(MG):,} | ddpca {len(DD):,} | atlas {len(AT):,}')

## 5. Checks and results
The zero-shot ESM-2 score is the free baseline every later model must beat (negative is correct: a low ESM score means a damaging mutation). The cross-fold table pools GRB2-SH3 and PSD95-PDZ3; GB1 drops out because its structure has no bound partner.

In [ ]:
esm_rho = {}
for s, col in [('ddpca', 'ddG_fold'), ('ddpca', 'ddG_bind'), ('megascale', 'ddG_fold')]:
    x = V[V.set == s].dropna(subset=[col, 'esm_score']); esm_rho[(s, col)] = (len(x), x.esm_score.corr(x[col], method='spearman'))
d = V[(V.set == 'ddpca') & V.has_structure & V.binding_reliable.fillna(False)]
XS = d.groupby(['buried', 'interface'])[['ddG_fold', 'ddG_bind']].median().reset_index()
XS.to_csv(R.out('cross_fold_separation.csv'), index=False)
a = V[V.set == 'atlas']
for k, v in [('variants_total', len(V)), ('variants_megascale', len(MG)), ('variants_ddpca', len(DD)),
             ('variants_atlas', len(AT)), ('dropped_by_residue_check', DROPPED), ('esm_domains', len(EMB)),
             ('esm_rho_ddpca_fold', esm_rho[('ddpca', 'ddG_fold')][1]), ('esm_rho_ddpca_bind', esm_rho[('ddpca', 'ddG_bind')][1]),
             ('esm_rho_megascale_fold', esm_rho[('megascale', 'ddG_fold')][1]),
             ('structure_rows', len(ST)), ('structure_domains', ST.domain_key.nunique()),
             ('alphafold_proteins_missing', len(AF_FAILED)),
             ('atlas_structure_coverage', a.has_structure.mean()), ('atlas_refmap_coverage', a.ref_pos.notna().mean()),
             ('atlas_groups', a.groupby(['uniprot', 'domain_idx']).ngroups)]:
    R.metric(k, v)
lab = lambda b, i: f"{'buried' if b else 'surface'}_{'interface' if i else 'not_interface'}"
for _, r in XS.iterrows():
    R.metric(f'xsep_{lab(r.buried, r.interface)}_fold', r.ddG_fold); R.metric(f'xsep_{lab(r.buried, r.interface)}_bind', r.ddG_bind)
R.section('Variant table', '', V.groupby('set').agg(variants=('pos', 'size'), domains=('domain_key', 'nunique')).reset_index())
R.section('Zero-shot ESM-2 against measured ΔΔG (Spearman)', 'Folding is tracked better than binding: the gap the thesis is about.',
          pd.DataFrame([dict(set=s, target=c, n=n, rho=r) for (s, c), (n, r) in esm_rho.items()]))
R.section('PDB offsets (derived, not assumed)', '', pd.DataFrame(offsets))
R.section('Cross-fold separation (GRB2-SH3 + PSD95-PDZ3, reliable binding only)', 'Replication of A1 across two folds.', XS, '{:.3f}')
R.section('AlphaFold proteins without a model', 'Very large proteins that AlphaFold DB splits into fragments; only F1 is requested.',
          pd.DataFrame({'uniprot': AF_FAILED}))
R.finish([f"{len(V):,} variants pass the residue check ({DROPPED} dropped): MegaScale {len(MG):,}, ddPCA {len(DD):,}, atlas {len(AT):,}.",
          f"ESM-2 embedded {len(EMB)} wild-type domains; each variant = 1,280 + 12 = 1,292 numbers.",
          f"Zero-shot ESM-2: folding ρ {esm_rho[('ddpca','ddG_fold')][1]:+.3f}, binding ρ {esm_rho[('ddpca','ddG_bind')][1]:+.3f} (ddPCA).",
          f"Atlas structure coverage {a.has_structure.mean():.1%}; reference-frame mapping {a.ref_pos.notna().mean():.1%}; "
          f"{len(AF_FAILED)} proteins have no AlphaFold model."])